# Features extraction - radiomics

#### Note:Environment with the most recent version of numpy and pyradiomics (upgrade in normal env)

In [1]:
!pip install radiomics

  Using cached radiomics-0.1.tar.gz (5.2 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for radiomics: filename=radiomics-0.1-py3-none-any.whl size=6097 sha256=6c0c263937782f29df04e43658dfb5ba5db97d4b5973181df0b9908b7cde51e0
  Stored in directory: c:\users\user\appdata\local\pip\cache\wheels\af\67\e0\4befebbb028e5cb8dc49d6a8c5eceda8605e2c3b1400680fd0
Successfully built radiomics


In [3]:
pip uninstall pyradiomics
pip install pyradiomics==3.0.1


^C
  Using cached pyradiomics-3.0.1.tar.gz (34.5 MB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for pyradiomics: filename=pyradiomics-3.0.1-cp38-cp38-win_amd64.whl size=122321 sha256=8ceefaa589b50efef81869e3af5ec76d5640c7eb0d54bbe46da6465982ac8b54
  Stored in directory: c:\users\user\appdata\local\pip\cache\wheels\df\be\06\5d09092e41d20673137f10ae62fb8d9da9adf14ce2552d7bea
Successfully built pyradiomics
  Attempting uninstall: pyradiomics
    Found existing installation: pyradiomics 3.1.1
    Uninstalling pyradiomics-3.1.1:
      Successfully uninstalled pyradiomics-3.1.1


In [1]:
import os
import six
import pickle
import pandas as pd
import radiomics
import SimpleITK as sitk
from radiomics import featureextractor
from functools import reduce
import numpy as np
from skimage.measure import find_contours
import pylidc as pl
from pylidc.utils import consensus
import matplotlib.pyplot as plt

### 2D features extraction

#### Config de fichier Params.yaml

In [7]:
with open("params.yaml", "w") as f:
    f.write("""
setting:
  binWidth: 25
  label: 1
  interpolator: 'sitkBSpline'
  resampledPixelSpacing:
  weightingNorm:
  force2D: True

imageType:
  Original: {}

featureClass:
  shape:
  firstorder: []
  glcm:
    - 'Autocorrelation'
    - 'JointAverage'
    - 'ClusterProminence'
    - 'ClusterShade'
    - 'ClusterTendency'
    - 'Contrast'
    - 'Correlation'
    - 'DifferenceAverage'
    - 'DifferenceEntropy'
    - 'DifferenceVariance'
    - 'JointEnergy'
    - 'JointEntropy'
    - 'Imc1'
    - 'Imc2'
    - 'Idm'
    - 'Idmn'
    - 'Id'
    - 'Idn'
    - 'InverseVariance'
    - 'MaximumProbability'
    - 'SumEntropy'
    - 'SumSquares'
  glrlm:
  glszm:
  gldm:
    """)


#### std_limit = 0.5

In [3]:
# Path were the images are stored
input_directory = "C:/Users/user/Desktop/PI_project/images/LIDC-IDRI"
# Path to the setup file from radiomics
params_file = "C:/Users/user/Desktop/PI_project/radiomics_config/Params.yaml"

path = "C:/Users/user/Desktop/PI_project/std_limit_0.5"

extractor = featureextractor.RadiomicsFeatureExtractor(params_file)

dic_list = []

# List of all the archives on the external drive
archives_npy = [archive for archive in os.listdir(path) if archive.endswith('cmask.npy')]

# Order the archives
ordered_archives = sorted(archives_npy, key=lambda x: int(x.split('_')[0]))

dic_list = []

for archive in ordered_archives:
    cmask = np.load(os.path.join(path, archive))
    image = sitk.GetImageFromArray(cmask.astype(float))

    features = extractor.execute(image, image, label=1)
    dic_list.append(features)

    nodule_id = int(archive.split('_')[0])
    features['Nodule_id'] = nodule_id

features = pd.DataFrame(dic_list)
csv_filename = os.path.join(path, 'features_radiomics_2d_0.5.csv')
features.to_csv(csv_filename, index=False)
features

,diagnostics_Versions_PyRadiomics,diagnostics_Versions_Numpy,diagnostics_Versions_SimpleITK,diagnostics_Versions_PyWavelet,diagnostics_Versions_Python,diagnostics_Configuration_Settings,diagnostics_Configuration_EnabledImageTypes,diagnostics_Image-original_Hash,diagnostics_Image-original_Dimensionality,diagnostics_Image-original_Spacing,...,original_gldm_GrayLevelVariance,original_gldm_HighGrayLevelEmphasis,original_gldm_LargeDependenceEmphasis,original_gldm_LargeDependenceHighGrayLevelEmphasis,original_gldm_LargeDependenceLowGrayLevelEmphasis,original_gldm_LowGrayLevelEmphasis,original_gldm_SmallDependenceEmphasis,original_gldm_SmallDependenceHighGrayLevelEmphasis,original_gldm_SmallDependenceLowGrayLevelEmphasis,Nodule_id
0,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},ea87a2a43bba24b4642a0005ebf9ca11496f042c,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,53.0228166797797,53.0228166797797,53.0228166797797,1.0,0.03007170459683713,0.03007170459683713,0.03007170459683713,3
1,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},5545209080d87a6226708c3ff5713bfd19e32633,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,18.3015873015873,18.3015873015873,18.3015873015873,1.0,0.09317019400352734,0.09317019400352734,0.09317019400352734,10
2,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},d15fc136149bf4d2c17594e6a293dc62a78372dc,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,24.096774193548388,24.096774193548388,24.096774193548388,1.0,0.06941756272401434,0.06941756272401434,0.06941756272401434,11
3,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},b0a099fd57d271966e68489ca302e08bd86d3a9e,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,27.416452442159382,27.416452442159382,27.416452442159382,1.0,0.06567937707788056,0.06567937707788056,0.06567937707788056,13
4,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},fed43996ffac93d06049abc9a961f1b5070ebfa6,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,49.151612903225804,49.151612903225804,49.151612903225804,1.0,0.04551056969233332,0.04551056969233332,0.04551056969233332,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
961,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},f6c599a2010a58190c6c21a48e99febc140f5fc0,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,25.464285714285715,25.464285714285715,25.464285714285715,1.0,0.056567460317460315,0.056567460317460315,0.056567460317460315,2642
962,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},9fdddde76941458c76678a677dc6cefd38f62ea0,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,20.897435897435898,20.897435897435898,20.897435897435898,1.0,0.06464387464387464,0.06464387464387464,0.06464387464387464,2644
963,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},c13756009931334f56e84d94771265e217f289d5,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,63.34114454859398,63.34114454859398,63.34114454859398,1.0,0.02349132226640405,0.02349132226640405,0.02349132226640405,2645
964,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},ca9d8b3a24039e84ad9ec11b01ffca322707e5be,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,27.025,27.025,27.025,1.0,0.056645833333333326,0.056645833333333326,0.056645833333333326,2646


#### std_limit = 1.0

In [4]:
# Path were the images are stored
input_directory = "C:/Users/user/Desktop/PI_project/images/LIDC-IDRI"
# Path to the setup file from radiomics
params_file = "C:/Users/user/Desktop/PI_project/radiomics_config/Params.yaml"

path = "C:/Users/user/Desktop/PI_project/std_limit_1.0"

extractor = featureextractor.RadiomicsFeatureExtractor(params_file)

dic_list = []

# List of all the archives on the external drive
archives_npy = [archive for archive in os.listdir(path) if archive.endswith('cmask.npy')]

# Order the archives
ordered_archives = sorted(archives_npy, key=lambda x: int(x.split('_')[0]))

dic_list = []

for archive in ordered_archives:
    cmask = np.load(os.path.join(path, archive))
    image = sitk.GetImageFromArray(cmask.astype(float))

    features = extractor.execute(image, image, label=1)
    dic_list.append(features)

    nodule_id = int(archive.split('_')[0])
    features['Nodule_id'] = nodule_id

features = pd.DataFrame(dic_list)
csv_filename = os.path.join(path, 'features_radiomics_2d_1.0.csv')
features.to_csv(csv_filename, index=False)
features


,diagnostics_Versions_PyRadiomics,diagnostics_Versions_Numpy,diagnostics_Versions_SimpleITK,diagnostics_Versions_PyWavelet,diagnostics_Versions_Python,diagnostics_Configuration_Settings,diagnostics_Configuration_EnabledImageTypes,diagnostics_Image-original_Hash,diagnostics_Image-original_Dimensionality,diagnostics_Image-original_Spacing,...,original_gldm_GrayLevelVariance,original_gldm_HighGrayLevelEmphasis,original_gldm_LargeDependenceEmphasis,original_gldm_LargeDependenceHighGrayLevelEmphasis,original_gldm_LargeDependenceLowGrayLevelEmphasis,original_gldm_LowGrayLevelEmphasis,original_gldm_SmallDependenceEmphasis,original_gldm_SmallDependenceHighGrayLevelEmphasis,original_gldm_SmallDependenceLowGrayLevelEmphasis,Nodule_id
0,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},6c9a580a252d49720868291e8202f6280702e325,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,68.86387875385911,68.86387875385911,68.86387875385911,1.0,0.020575917587022128,0.020575917587022128,0.020575917587022128,2
1,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},ea87a2a43bba24b4642a0005ebf9ca11496f042c,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,53.0228166797797,53.0228166797797,53.0228166797797,1.0,0.03007170459683713,0.03007170459683713,0.03007170459683713,3
2,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},87deef3afa40c86ea2a0f2f9220b32301bbddfd3,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,32.7410071942446,32.7410071942446,32.7410071942446,1.0,0.04801748510488661,0.04801748510488661,0.04801748510488661,9
3,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},5545209080d87a6226708c3ff5713bfd19e32633,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,18.3015873015873,18.3015873015873,18.3015873015873,1.0,0.09317019400352734,0.09317019400352734,0.09317019400352734,10
4,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},d15fc136149bf4d2c17594e6a293dc62a78372dc,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,24.096774193548388,24.096774193548388,24.096774193548388,1.0,0.06941756272401434,0.06941756272401434,0.06941756272401434,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1651,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},f6c599a2010a58190c6c21a48e99febc140f5fc0,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,25.464285714285715,25.464285714285715,25.464285714285715,1.0,0.056567460317460315,0.056567460317460315,0.056567460317460315,2642
1652,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},9fdddde76941458c76678a677dc6cefd38f62ea0,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,20.897435897435898,20.897435897435898,20.897435897435898,1.0,0.06464387464387464,0.06464387464387464,0.06464387464387464,2644
1653,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},c13756009931334f56e84d94771265e217f289d5,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,63.34114454859398,63.34114454859398,63.34114454859398,1.0,0.02349132226640405,0.02349132226640405,0.02349132226640405,2645
1654,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},ca9d8b3a24039e84ad9ec11b01ffca322707e5be,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,27.025,27.025,27.025,1.0,0.056645833333333326,0.056645833333333326,0.056645833333333326,2646


#### std_limit = 1.5

In [5]:
# Path were the images are stored
input_directory = "C:/Users/user/Desktop/PI_project/images/LIDC-IDRI"
# Path to the setup file from radiomics
params_file = "C:/Users/user/Desktop/PI_project/radiomics_config/Params.yaml"

path = "C:/Users/user/Desktop/PI_project/std_limit_1.5"

extractor = featureextractor.RadiomicsFeatureExtractor(params_file)

extractor = featureextractor.RadiomicsFeatureExtractor(params_file)

dic_list = []

# List of all the archives on the external drive
archives_npy = [archive for archive in os.listdir(path) if archive.endswith('cmask.npy')]

# Order the archives
ordered_archives = sorted(archives_npy, key=lambda x: int(x.split('_')[0]))

dic_list = []

for archive in ordered_archives:
    cmask = np.load(os.path.join(path, archive))
    image = sitk.GetImageFromArray(cmask.astype(float))

    features = extractor.execute(image, image, label=1)
    dic_list.append(features)

    nodule_id = int(archive.split('_')[0])
    features['Nodule_id'] = nodule_id

features = pd.DataFrame(dic_list)
csv_filename = os.path.join(path, 'features_radiomics_2d_1.5.csv')
features.to_csv(csv_filename, index=False)
features


,diagnostics_Versions_PyRadiomics,diagnostics_Versions_Numpy,diagnostics_Versions_SimpleITK,diagnostics_Versions_PyWavelet,diagnostics_Versions_Python,diagnostics_Configuration_Settings,diagnostics_Configuration_EnabledImageTypes,diagnostics_Image-original_Hash,diagnostics_Image-original_Dimensionality,diagnostics_Image-original_Spacing,...,original_gldm_GrayLevelVariance,original_gldm_HighGrayLevelEmphasis,original_gldm_LargeDependenceEmphasis,original_gldm_LargeDependenceHighGrayLevelEmphasis,original_gldm_LargeDependenceLowGrayLevelEmphasis,original_gldm_LowGrayLevelEmphasis,original_gldm_SmallDependenceEmphasis,original_gldm_SmallDependenceHighGrayLevelEmphasis,original_gldm_SmallDependenceLowGrayLevelEmphasis,Nodule_id
0,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},3dbcca2f07a28dd238aeea62535e787abbfeda44,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,61.29513633014002,61.29513633014002,61.29513633014002,1.0,0.024672632479600103,0.024672632479600103,0.024672632479600103,1
1,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},6c9a580a252d49720868291e8202f6280702e325,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,68.86387875385911,68.86387875385911,68.86387875385911,1.0,0.020575917587022128,0.020575917587022128,0.020575917587022128,2
2,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},ea87a2a43bba24b4642a0005ebf9ca11496f042c,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,53.0228166797797,53.0228166797797,53.0228166797797,1.0,0.03007170459683713,0.03007170459683713,0.03007170459683713,3
3,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},8a8f88ba678db54be433931d9507c7b26f4e4f17,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,60.239123727244674,60.239123727244674,60.239123727244674,1.0,0.023493203877023174,0.023493203877023174,0.023493203877023174,4
4,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},ccaf09e8511dca39e32d79a8e9e9dd856472b0ff,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,40.8007662835249,40.8007662835249,40.8007662835249,1.0,0.038992273935258795,0.038992273935258795,0.038992273935258795,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2435,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},c13756009931334f56e84d94771265e217f289d5,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,63.34114454859398,63.34114454859398,63.34114454859398,1.0,0.02349132226640405,0.02349132226640405,0.02349132226640405,2645
2436,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},ca9d8b3a24039e84ad9ec11b01ffca322707e5be,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,27.025,27.025,27.025,1.0,0.056645833333333326,0.056645833333333326,0.056645833333333326,2646
2437,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},aa28668ac61b24d0fb8627df9fab91ddcf0843f0,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,58.2309846010266,58.2309846010266,58.2309846010266,1.0,0.02491632002604882,0.02491632002604882,0.02491632002604882,2648
2438,v3.0.1,1.20.3,2.4.1,1.4.1,3.8.20,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},347615a17f395f599e2ed65b6c358a55f4dc744e,3D,"(1.0, 1.0, 1.0)",...,0.0,1.0,50.11631944444444,50.11631944444444,50.11631944444444,1.0,0.03298220743094048,0.03298220743094048,0.03298220743094048,2649


### 3D features extration

#### Due to the lack of storage this part of the code can not be executed 

#### Configuração do ficheiro Params.yaml

In [6]:
#Settings to use, possible settings are listed in the documentation (section "Customizing the extraction").
setting:
  binWidth: 25
  label: 1
  interpolator: 'sitkBSpline' # This is an enumerated value, here None is not allowed
  resampledPixelSpacing: # This disables resampling, as it is interpreted as None, to enable it, specify spacing in x, y, z as [x, y , z]
  #verbose: True
  weightingNorm: # If no value is specified, it is interpreted as None
  force2D: False  # Esta configuração ativa a extração de características 2D

#Image types to use: "Original" for unfiltered image, for possible filters, see documentation.
imageType:
  Original: {} # for dictionaries / mappings, None values are not allowed, '{}' is interpreted as an empty dictionary

#Featureclasses, from which features must be calculated. If a featureclass is not mentioned, no features are calculated
#for that class. Otherwise, the specified features are calculated, or, if none are specified, all are calculated (excluding redundant/deprecated features).
featureClass:
  shape:
  firstorder: [] # specifying an empty list has the same effect as specifying nothing.
  glcm:  # Disable SumAverage by specifying all other GLCM features available
    - 'Autocorrelation'
    - 'JointAverage'
    - 'ClusterProminence'
    - 'ClusterShade'
    - 'ClusterTendency'
    - 'Contrast'
    - 'Correlation'
    - 'DifferenceAverage'
    - 'DifferenceEntropy'
    - 'DifferenceVariance'
    - 'JointEnergy'
    - 'JointEntropy'
    - 'Imc1'
    - 'Imc2'
    - 'Idm'
    - 'Idmn'
    - 'Id'
    - 'Idn'
    - 'InverseVariance'
    - 'MaximumProbability'
    - 'SumEntropy'
    - 'SumSquares'
  glrlm: # for lists none values are allowed, in this case, all features are enabled
  glszm:
  gldm:  # contains deprecated features, but as no individual features are specified, the deprecated features are not enabled


SyntaxError: invalid syntax (993684486.py, line 2)

#### std_limit = 0.5

In [10]:
import os
import numpy as np
import SimpleITK as sitk
import pandas as pd
from radiomics import featureextractor

# Define paths
input_directory = "C:/Users/user/Desktop/PI_project/images/LIDC-IDRI"
params_file = "C:/Users/user/Desktop/PI_project/radiomics_config/Params3D.yaml"
path = "C:/Users/user/Desktop/PI_project/std_limit_0.5"

# Initialize the radiomics feature extractor
extractor = featureextractor.RadiomicsFeatureExtractor(params_file)

# List to store features
dic_list = []

# List of all the archives on the external drive (only .npy files)
archives_npy = [archive for archive in os.listdir(path) if archive.endswith('volume.npy')]

# Order the archives by the first part of their filenames (assuming the nodule_id is the first element)
ordered_archives = sorted(archives_npy, key=lambda x: int(x.split('_')[0]))

# Iterate over the ordered archives
for archive in ordered_archives:
    print(f"Processing {archive}...")  # Debug statement to track progress
    
    # Load the volume (image) data from the .npy file
    volume = np.load(os.path.join(path, archive))
    
    # Check if the volume is empty
    if volume.size == 0:
        print(f"Empty volume in {archive}. Skipping.")
        continue
    
    # Convert numpy array to SimpleITK image
    image = sitk.GetImageFromArray(volume.astype(float))

    # Extract features (assuming label=1, modify if necessary)
    features = extractor.execute(image, image, label=1)
    
    # Add nodule_id to the features dictionary
    nodule_id = int(archive.split('_')[0])
    features['Nodule_id'] = nodule_id

    # Append the extracted features to the list
    dic_list.append(features)

# Convert the list of features dictionaries into a pandas DataFrame
features_df = pd.DataFrame(dic_list)

# Specify the output CSV file path
csv_filename = os.path.join(path, 'features_radiomics_3d_0.5.csv')

# Write the DataFrame to CSV
features_df.to_csv(csv_filename, index=False)

# Print a message and return the DataFrame for further verification
print(f"Features written to {csv_filename}")
features_df


Features written to C:/Users/user/Desktop/PI_project/std_limit_0.5\features_radiomics_3d_0.5.csv


""


#### std_limit = 1.0

In [14]:
import os
import numpy as np
import SimpleITK as sitk
import pandas as pd
from radiomics import featureextractor

# Define paths
input_directory = "C:/Users/user/Desktop/PI_project/images/LIDC-IDRI"
params_file = "C:/Users/user/Desktop/PI_project/radiomics_config/Params3D.yaml"
path = "C:/Users/user/Desktop/PI_project/std_limit_1.0"

# Initialize the radiomics feature extractor
extractor = featureextractor.RadiomicsFeatureExtractor(params_file)

# List to store features
dic_list = []

# List of all the archives on the specified path (only .npy files)
archives_npy = [archive for archive in os.listdir(path) if archive.endswith('volume.npy')]

# Order the archives by the first part of their filenames (assuming the nodule_id is the first element)
ordered_archives = sorted(archives_npy, key=lambda x: int(x.split('_')[0]))

# Iterate over the ordered archives
for archive in ordered_archives:
    print(f"Processing {archive}...")  # Debug statement to track progress
    
    # Load the volume (image) data from the .npy file
    volume = np.load(os.path.join(path, archive))
    
    # Check if the volume is empty
    if volume.size == 0:
        print(f"Empty volume in {archive}. Skipping.")
        continue
    
    # Convert numpy array to SimpleITK image
    image = sitk.GetImageFromArray(volume.astype(float))

    # Extract features (assuming label=1, modify if necessary)
    features = extractor.execute(image, image, label=1)
    
    # Add nodule_id to the features dictionary
    nodule_id = int(archive.split('_')[0])
    features['Nodule_id'] = nodule_id

    # Append the extracted features to the list
    dic_list.append(features)

# Convert the list of features dictionaries into a pandas DataFrame
features_df = pd.DataFrame(dic_list)

# Specify the output CSV file path
csv_filename = os.path.join(path, 'features_radiomics_3d_1.0.csv')

# Write the DataFrame to CSV
features_df.to_csv(csv_filename, index=False)

# Print a message and return the DataFrame for further verification
print(f"Features written to {csv_filename}")
features_df


Features written to C:/Users/user/Desktop/PI_project/std_limit_1.0\features_radiomics_3d_1.0.csv


""


#### std_limit = 1.5

In [15]:
import os
import numpy as np
import SimpleITK as sitk
import pandas as pd
from radiomics import featureextractor


# Define paths
input_directory = "C:/Users/user/Desktop/PI_project/images/LIDC-IDRI"
params_file = "C:/Users/user/Desktop/PI_project/radiomics_config/Params3D.yaml"
path = "C:/Users/user/Desktop/PI_project/std_limit_1.5"

# Initialize the radiomics feature extractor
extractor = featureextractor.RadiomicsFeatureExtractor(params_file)

# List to store features
dic_list = []

# List of all the archives on the specified path (only .npy files)
archives_npy = [archive for archive in os.listdir(path) if archive.endswith('volume.npy')]

# Order the archives by the first part of their filenames (assuming the nodule_id is the first element)
ordered_archives = sorted(archives_npy, key=lambda x: int(x.split('_')[0]))

# Iterate over the ordered archives
for archive in ordered_archives:
    print(f"Processing {archive}...")  # Debug statement to track progress
    
    # Load the volume (image) data from the .npy file
    volume = np.load(os.path.join(path, archive))
    
    # Check if the volume is empty
    if volume.size == 0:
        print(f"Empty volume in {archive}. Skipping.")
        continue
    
    # Convert numpy array to SimpleITK image
    image = sitk.GetImageFromArray(volume.astype(float))

    # Extract features (assuming label=1, modify if necessary)
    features = extractor.execute(image, image, label=1)
    
    # Add nodule_id to the features dictionary
    nodule_id = int(archive.split('_')[0])
    features['Nodule_id'] = nodule_id

    # Append the extracted features to the list
    dic_list.append(features)

# Convert the list of features dictionaries into a pandas DataFrame
features_df = pd.DataFrame(dic_list)

# Specify the output CSV file path
csv_filename = os.path.join(path, 'features_radiomics_3d_1.5.csv')

# Write the DataFrame to CSV
features_df.to_csv(csv_filename, index=False)

# Print a message and return the DataFrame for further verification
print(f"Features written to {csv_filename}")
features_df


Features written to C:/Users/user/Desktop/PI_project/std_limit_1.5\features_radiomics_3d_1.5.csv


""
